In [ ]:
import os 
import sys 
import torch 
import torch .nn as nn 
import torch .optim as optim 
from datetime import datetime 
from pathlib import Path

current_dir = Path.cwd().resolve()
for parent in [current_dir, *current_dir.parents]:
    if (parent / "src").exists():
        project_root = str(parent)
        break
else:
    project_root = str(current_dir)
if project_root not in sys .path :
    sys .path .insert (0 ,project_root )

from src .utils .setup import setup_device ,setup_directories 
from src .utils .dataloader import BreastMNISTLoader ,BCDRLoader ,INbreastLoader ,BUSUCLMLoader 
from src .utils .training import training_loop ,test_model 
from src .models .main .classical import ClassicalModel ,ShallowCNN 

print ("✓ Imports successful")

In [ ]:

CONFIG ={

'model_type':'shallow',
'num_classes':2 ,
'input_channels':1 ,


'dataset':'breastmnist',
'batch_size':16 ,
'image_size':224 ,


'epochs':100 ,
'learning_rate':1e-3 ,
}

print ("Configuration:")
for key ,value in CONFIG .items ():
    print (f"  {key }: {value }")

In [ ]:
device =setup_device ()
dirs =setup_directories ()


dataset_mapping ={
'breastmnist':'BreastMNIST',
'bcdr':'BCDR',
'inbreast':'INbreast',
'busuclm':'BUSUCLM'
}
dataset_name =dataset_mapping .get (CONFIG ['dataset'],'BreastMNIST')
model_type =CONFIG ['model_type']
save_dir =os .path .join (
dirs .get ("project_dir","."),
"experiments","models",dataset_name ,"classical",model_type 
)
os .makedirs (save_dir ,exist_ok =True )

print (f"Device: {device }")
print (f"Save directory: {save_dir }")

In [ ]:
if CONFIG ['dataset']=='bcdr':
    loader =BCDRLoader (
    batch_size =CONFIG ['batch_size'],
    image_size =CONFIG ['image_size'],
    root_dir ='src/data/BCDR/png'
    )
    print (f"✓ BCDR dataset loaded")
elif CONFIG ['dataset']=='inbreast':
    loader =INbreastLoader (
    batch_size =CONFIG ['batch_size'],
    image_size =CONFIG ['image_size'],
    root_dir ='src/data/INbreast/png'
    )
    print (f"✓ INbreast dataset loaded")
elif CONFIG ['dataset']=='busuclm':
    loader =BUSUCLMLoader (
    batch_size =CONFIG ['batch_size'],
    image_size =CONFIG ['image_size'],
    root_dir ='src/data/BUS-UCLM/png'
    )
    print (f"✓ BUSUCLM dataset loaded")
else :
    loader =BreastMNISTLoader (
    batch_size =CONFIG ['batch_size'],
    image_size =CONFIG ['image_size'],
    )
    print (f"✓ BreastMNIST dataset loaded")

train_loader ,test_loader ,val_loader =loader .get_dataloaders ()

print (f"  Training batches: {len (train_loader )}")
print (f"  Validation batches: {len (val_loader )}")
print (f"  Test batches: {len (test_loader )}")

In [ ]:
if CONFIG ['model_type']=='shallow':
    model =ShallowCNN (
    num_classes =CONFIG ['num_classes'],
    input_channels =CONFIG ['input_channels']
    ).to (device )
    model_name ='shallow_no_encoder'
else :
    model =ClassicalModel (
    num_classes =CONFIG ['num_classes']
    ).to (device )
    model_name ='resnet'

total_params =sum (p .numel ()for p in model .parameters ())
trainable_params =sum (p .numel ()for p in model .parameters ()if p .requires_grad )

print (f"✓ Model created: {model .__class__ .__name__ }")
print (f"  Total parameters: {total_params :,}")
print (f"  Trainable parameters: {trainable_params :,}")
print (f"  All parameters are trainable (no frozen encoder)")

In [ ]:
criterion =nn .CrossEntropyLoss ()
optimizer =optim .Adam (model .parameters (),lr =CONFIG ['learning_rate'])

print (f"✓ Training setup complete")
print (f"  Loss: CrossEntropyLoss")
print (f"  Optimizer: Adam (lr={CONFIG ['learning_rate']})")
print (f"  Epochs: {CONFIG ['epochs']}")

In [ ]:
print (f"\n{'='*80 }")
print (f"Starting Training - {datetime .now ().strftime ('%Y-%m-%d %H:%M:%S')}")
print (f"{'='*80 }\n")

history =training_loop (
model =model ,
train_loader =train_loader ,
val_loader =val_loader ,
criterion =criterion ,
optimizer =optimizer ,
device =device ,
num_epochs =CONFIG ['epochs'],
save_dir =save_dir ,
model_name =model_name ,
extract_features =False ,
save_best_only =True ,
save_all_epochs =False 
)

print (f"\n{'='*80 }")
print (f"Training Complete - {datetime .now ().strftime ('%Y-%m-%d %H:%M:%S')}")
print (f"{'='*80 }")

In [ ]:

model_path =os.path .join (save_dir ,f"best_{model_name }.pth")

if os.path.isfile (model_path ):
    state =torch .load (model_path ,map_location =device )
    model .load_state_dict (state )
    print (f"✓ Loaded best model from {model_path }")
else :
    print ("⚠ Warning: No best model found, using final epoch weights")

model .eval ()

In [ ]:

print (f"\n{'='*80 }")
print ("Testing Best Model")
print (f"{'='*80 }\n")

test_result =test_model (
model =model ,
dataloader =test_loader ,
criterion =criterion ,
device =device ,
hyperparams =CONFIG 
)


test_loss ,test_acc ,test_prec, test_rec, test_f1, test_auc = test_result 


test_metrics ={
'loss':float (test_loss),
'accuracy':float (test_acc),
'precision':float (test_prec),
'recall':float (test_rec),
'f1':float (test_f1),
'auc':float (test_auc)
}

print ("\nTest Results:")
print (f"  Loss: {test_metrics ['loss']:.4f}")
print (f"  Accuracy: {test_metrics ['accuracy']:.4f}")
print (f"  Precision: {test_metrics ['precision']:.4f}")
print (f"  Recall: {test_metrics ['recall']:.4f}")
print (f"  F1 Score: {test_metrics ['f1']:.4f}")
print (f"  AUC: {test_metrics ['auc']:.4f}")

In [ ]:

results_file =os.path.join (save_dir, f'test_results_{model_name }.txt')

with open (results_file ,'w',encoding ='utf-8')as f :
    f.write ("="*80 +"\n")
    f.write (f"TEST RESULTS - Classical Model WITHOUT Encoder\n")
    f.write ("="*80 +"\n\n")

    f.write (f"Timestamp: {datetime.now().strftime ('%Y-%m-%d %H:%M:%S')}\n\n")

    f.write ("MODEL CONFIGURATION\n")
    f.write ("-"*80 +"\n")
    f.write (f"Model Type: {model .__class__ .__name__ }\n")
    f.write (f"Model Variant: {CONFIG ['model_type']}\n")
    f.write (f"Dataset: {CONFIG ['dataset'].upper ()}\n")
    f.write (f"Encoder: None (end-to-end training)\n")
    f.write (f"Total Parameters: {total_params :,}\n")
    f.write (f"Trainable Parameters: {trainable_params :,}\n\n")

    f.write ("TRAINING HYPERPARAMETERS\n")
    f.write ("-"*80 +"\n")
    f.write (f"Epochs: {CONFIG ['epochs']}\n")
    f.write (f"Batch Size: {CONFIG ['batch_size']}\n")
    f.write (f"Learning Rate: {CONFIG ['learning_rate']}\n")
    f.write (f"Image Size: {CONFIG ['image_size']}\n")
    f.write (f"Input Channels: {CONFIG ['input_channels']}\n")
    f.write (f"Number of Classes: {CONFIG ['num_classes']}\n\n")

    f.write ("BEST VALIDATION METRICS\n")
    f.write ("-"*80 +"\n")
    f.write (f"Best Validation Accuracy: {max (history ['val_acc']):.4f}\n")
    f.write (f"Best Validation F1 Score: {max (history ['val_f1']):.4f}\n")
    f.write (f"Best Validation AUC: {max (history ['val_auc']):.4f}\n")
    f.write (f"Best Validation Precision: {max (history ['val_precision']):.4f}\n")
    f.write (f"Best Validation Recall: {max (history ['val_recall']):.4f}\n\n")

    f.write ("FINAL TEST METRICS\n")
    f.write ("-"*80 +"\n")
    f.write (f"Test Loss: {test_metrics ['loss']:.4f}\n")
    f.write (f"Test Accuracy: {test_metrics ['accuracy']:.4f}\n")
    f.write (f"Test Precision: {test_metrics ['precision']:.4f}\n")
    f.write (f"Test Recall: {test_metrics ['recall']:.4f}\n")
    f.write (f"Test F1 Score: {test_metrics ['f1']:.4f}\n")
    f.write (f"Test AUC: {test_metrics ['auc']:.4f}\n\n")

    if 'macro_precision'in test_metrics :
        f.write ("MACRO-AVERAGED METRICS\n")
        f.write ("-"*80 +"\n")
        f.write (f"Macro Precision: {test_metrics ['macro_precision']:.4f}\n")
        f.write (f"Macro Recall: {test_metrics ['macro_recall']:.4f}\n")
        f.write (f"Macro F1 Score: {test_metrics ['macro_f1']:.4f}\n\n")

    if 'micro_precision'in test_metrics :
        f.write ("MICRO-AVERAGED METRICS\n")
        f.write ("-"*80 +"\n")
        f.write (f"Micro Precision: {test_metrics ['micro_precision']:.4f}\n")
        f.write (f"Micro Recall: {test_metrics ['micro_recall']:.4f}\n")
        f.write (f"Micro F1 Score: {test_metrics ['micro_f1']:.4f}\n\n")

    f.write ("MODEL SAVE LOCATION\n")
    f.write ("-"*80 +"\n")
    f.write (f"Directory: {save_dir }\n")
    f.write (f"Best Model: best_{model_name }.pth\n\n")

    f.write ("="*80 +"\n")
    f.write ("END OF REPORT\n")
    f.write ("="*80 +"\n")

print (f"✓ Test results saved to: {results_file }")